| Pseudo Intent | Count |
| :--- | :--- |
| Delivery Issue | 4,046 |
| Other / Unclear | 2,274 |
| Account / Access / Security | 633 |
| Order Management | 624 |
| Product / Device Support | 605 |
| Payment / Billing | 537 |
| Return / Refund | 475 |
| Damaged / Wrong / Missing Item | 458 |
| Digital Content | 292 |
| Promotion / Gift Card / Credit | 40 |
| Prime Membership | 16 |

Saved to `data/processed/dev_pseudo_labeled.csv`

In [50]:
# ============================================================
# PSEUDO-LABEL AUDIT - FINAL ANALYSIS
# ============================================================

import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report

AUDIT_PATH = "../data/golden/pseudo_label_audit_100.csv"

# ------------------------------------------------------------
# 1. Load audit
# ------------------------------------------------------------

audit = pd.read_csv(AUDIT_PATH)

print("✅ Audit loaded")
print("Shape:", audit.shape)

# ------------------------------------------------------------
# 2. Validate required columns
# ------------------------------------------------------------

required_columns = [
    "root_tweet_id",
    "pseudo_intent",
    "human_label"
]

missing_columns = [
    col for col in required_columns
    if col not in audit.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

# ------------------------------------------------------------
# 3. Check human labels
# ------------------------------------------------------------

filled_human_labels = (
    audit["human_label"]
    .notna()
    .sum()
)

print("\nHuman labels filled:", filled_human_labels)

if filled_human_labels < len(audit):
    raise ValueError(
        f"""
Only {filled_human_labels}/{len(audit)} human labels exist.

The audit cannot be evaluated yet because some
human labels are missing.
"""
    )

# ------------------------------------------------------------
# 4. Clean labels
# ------------------------------------------------------------

audit["human_label"] = (
    audit["human_label"]
    .astype(str)
    .str.strip()
)

audit["pseudo_intent"] = (
    audit["pseudo_intent"]
    .astype(str)
    .str.strip()
)

# ------------------------------------------------------------
# 5. Calculate correctness
# ------------------------------------------------------------

audit["correct"] = (
    audit["human_label"]
    == audit["pseudo_intent"]
)

total = len(audit)
correct = int(audit["correct"].sum())
incorrect = total - correct

accuracy = correct / total * 100

# ------------------------------------------------------------
# 6. Overall result
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PSEUDO-LABEL AUDIT RESULT")
print("=" * 70)

print(f"Audit examples       : {total}")
print(f"Correct predictions  : {correct}")
print(f"Incorrect predictions: {incorrect}")
print(f"Audit accuracy       : {accuracy:.2f}%")

# ------------------------------------------------------------
# 7. Confusion matrix
# ------------------------------------------------------------

labels = sorted(
    set(audit["human_label"]) |
    set(audit["pseudo_intent"])
)

cm = confusion_matrix(
    audit["human_label"],
    audit["pseudo_intent"],
    labels=labels
)

cm_df = pd.DataFrame(
    cm,
    index=labels,
    columns=labels
)

cm_df.index.name = "Human Label"
cm_df.columns.name = "AI Label"

print("\n" + "=" * 70)
print("CONFUSION MATRIX")
print("=" * 70)

display(cm_df)

# ------------------------------------------------------------
# 8. Classification report
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CLASSIFICATION REPORT")
print("=" * 70)

report = classification_report(
    audit["human_label"],
    audit["pseudo_intent"],
    labels=labels,
    target_names=labels,
    zero_division=0
)

print(report)

# ------------------------------------------------------------
# 9. Per-intent accuracy
# ------------------------------------------------------------

per_intent = []

for label in labels:

    subset = audit[
        audit["human_label"] == label
    ]

    samples = len(subset)
    correct_count = int(subset["correct"].sum())

    per_intent.append({
        "Intent": label,
        "Samples": samples,
        "Correct": correct_count,
        "Incorrect": samples - correct_count,
        "Accuracy %": round(
            correct_count / samples * 100,
            2
        )
    })

per_intent_df = pd.DataFrame(per_intent)

print("\n" + "=" * 70)
print("PER-INTENT PERFORMANCE")
print("=" * 70)

display(
    per_intent_df.sort_values(
        "Accuracy %",
        ascending=True
    )
)

# ------------------------------------------------------------
# 10. Show disagreements
# ------------------------------------------------------------

errors = audit[
    ~audit["correct"]
].copy()

print("\n" + "=" * 70)
print("DISAGREEMENTS")
print("=" * 70)

print("Total disagreements:", len(errors))

if len(errors) > 0:

    display(
        errors[
            [
                "root_tweet_id",
                "human_label",
                "pseudo_intent",
                "conversation"
            ]
        ]
    )

# ------------------------------------------------------------
# 11. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL AUDIT SUMMARY")
print("=" * 70)

print(f"""
Audit size       : {total}
Correct          : {correct}
Incorrect        : {incorrect}
Agreement        : {accuracy:.2f}%

This measures agreement between the Groq
pseudo-labels and human labels on the 100-example
audit sample.


""")

✅ Audit loaded
Shape: (100, 5)

Human labels filled: 100

PSEUDO-LABEL AUDIT RESULT
Audit examples       : 100
Correct predictions  : 89
Incorrect predictions: 11
Audit accuracy       : 89.00%

CONFUSION MATRIX


AI Label,Account / Access / Security,Damaged / Wrong / Missing Item,Delivery Issue,Digital Content,Order Management,Other / Unclear,Payment / Billing,Prime Membership,Product / Device Support,Return / Refund
Human Label,,,,,,,,,,
Account / Access / Security,10,0,1,0,0,0,0,0,0,0
Damaged / Wrong / Missing Item,0,3,0,0,0,0,0,0,0,0
Delivery Issue,0,0,29,0,0,1,0,0,0,0
Digital Content,0,0,0,4,0,0,0,0,0,0
Order Management,0,1,2,0,3,0,0,0,0,0
Other / Unclear,0,0,0,0,0,24,0,0,0,0
Payment / Billing,0,0,2,0,0,0,7,0,0,0
Prime Membership,0,0,2,0,0,0,0,1,0,0
Product / Device Support,0,0,0,0,0,1,0,0,4,0



CLASSIFICATION REPORT
                                precision    recall  f1-score   support

   Account / Access / Security       1.00      0.91      0.95        11
Damaged / Wrong / Missing Item       0.75      1.00      0.86         3
                Delivery Issue       0.78      0.97      0.87        30
               Digital Content       1.00      1.00      1.00         4
              Order Management       1.00      0.50      0.67         6
               Other / Unclear       0.92      1.00      0.96        24
             Payment / Billing       1.00      0.78      0.88         9
              Prime Membership       1.00      0.33      0.50         3
      Product / Device Support       1.00      0.80      0.89         5
               Return / Refund       1.00      0.80      0.89         5

                      accuracy                           0.89       100
                     macro avg       0.95      0.81      0.85       100
                  weighted avg       0.

,Intent,Samples,Correct,Incorrect,Accuracy %
7,Prime Membership,3,1,2,33.33
4,Order Management,6,3,3,50.00
6,Payment / Billing,9,7,2,77.78
8,Product / Device Support,5,4,1,80.00
9,Return / Refund,5,4,1,80.00
0,Account / Access / Security,11,10,1,90.91
2,Delivery Issue,30,29,1,96.67
1,Damaged / Wrong / Missing Item,3,3,0,100.00
5,Other / Unclear,24,24,0,100.00
3,Digital Content,4,4,0,100.00



DISAGREEMENTS
Total disagreements: 11


,root_tweet_id,human_label,pseudo_intent,conversation
3,1085150,Product / Device Support,Other / Unclear,CUSTOMER: Can a non-profit list itself as its ...
18,613997,Prime Membership,Delivery Issue,CUSTOMER: Amazonの prime now届いたー！\n多12時58分に頼んで1...
21,1004594,Order Management,Delivery Issue,CUSTOMER: It REALLY annoys me that each and ev...
25,2317390,Delivery Issue,Other / Unclear,CUSTOMER: .@115821 @115850 request you to not ...
40,1097715,Order Management,Damaged / Wrong / Missing Item,CUSTOMER: Wait! It isnt my order #catstagram #...
44,132376,Payment / Billing,Delivery Issue,CUSTOMER: Amazon prime will be the death of my...
54,2500688,Return / Refund,Delivery Issue,CUSTOMER: @115821 pretty sure it's illegal to ...
57,676933,Prime Membership,Delivery Issue,CUSTOMER: Amazon prime student 有料会員になってしまった\n腐...
62,1376064,Account / Access / Security,Delivery Issue,CUSTOMER: I actually cancelled @115821 prime t...
65,1540467,Order Management,Delivery Issue,CUSTOMER: Time to cancel @115821 prime until u...



FINAL AUDIT SUMMARY

Audit size       : 100
Correct          : 89
Incorrect        : 11
Agreement        : 89.00%

This measures agreement between the Groq
pseudo-labels and human labels on the 100-example
audit sample.





In [49]:
# ============================================================
# SHOW ALL PSEUDO-LABEL DISAGREEMENTS
# ============================================================

disagreements = audit[
    audit["pseudo_intent"].str.strip() != audit["human_label"].str.strip()
].copy()

print(f"Total disagreements: {len(disagreements)}")
print("=" * 80)

for _, row in disagreements.iterrows():
    print(f"\nROOT TWEET ID : {row['root_tweet_id']}")
    print(f"AI LABEL      : {row['pseudo_intent']}")
    print(f"HUMAN LABEL   : {row['human_label']}")
    print("-" * 80)
    print(row["conversation"])
    print("=" * 80)

Total disagreements: 11

ROOT TWEET ID : 1085150
AI LABEL      : Other / Unclear
HUMAN LABEL   : Product / Device Support
--------------------------------------------------------------------------------
CUSTOMER: Can a non-profit list itself as its own beneficiary on @115821 smile? #nonprofit @AmazonHelp

AMAZON: @375966 Thanks for reaching out! When you have the time, please contact our AmazonSmile Support team here: https://t.co/8J2Xg4zvTd ^KN

ROOT TWEET ID : 613997
AI LABEL      : Delivery Issue
HUMAN LABEL   : Prime Membership
--------------------------------------------------------------------------------
CUSTOMER: Amazonの prime now届いたー！
多12時58分に頼んで13時31分出荷、15時13分に受け取りました。ありがたい！わたしは市内住みでは無いのでセンターから10㌔以上離れていますがこの迅速さ。配達の方も礼儀正しくて好印象でした。また急ぎたい時におねがいします。 https://t.co/HBs0H21xwU

AMAZON: @203475 Prime Nowをご利用いただき、ありがとうございました！またお役に立てれば嬉しいです( *´艸｀) EK

ROOT TWEET ID : 1004594
AI LABEL      : Delivery Issue
HUMAN LABEL   : Order Management
-------------------------------------------------

In [51]:
# ============================================================
# DISAGREEMENT SUMMARY
# ============================================================

from collections import Counter

# Count each AI → Human transition
error_pairs = Counter(
    zip(
        disagreements["pseudo_intent"].str.strip(),
        disagreements["human_label"].str.strip()
    )
)

print("=" * 80)
print("AI LABEL → HUMAN LABEL ERROR PATTERNS")
print("=" * 80)

for (ai_label, human_label), count in error_pairs.most_common():
    print(f"{count:2d}  {ai_label}  →  {human_label}")

print("\n" + "=" * 80)
print("ERRORS BY AI PREDICTION")
print("=" * 80)

for label, count in Counter(disagreements["pseudo_intent"]).most_common():
    print(f"{count:2d}  {label}")

print("\n" + "=" * 80)
print("ERRORS BY HUMAN LABEL")
print("=" * 80)

for label, count in Counter(disagreements["human_label"]).most_common():
    print(f"{count:2d}  {label}")

AI LABEL → HUMAN LABEL ERROR PATTERNS
 2  Delivery Issue  →  Prime Membership
 2  Delivery Issue  →  Order Management
 2  Delivery Issue  →  Payment / Billing
 1  Other / Unclear  →  Product / Device Support
 1  Other / Unclear  →  Delivery Issue
 1  Damaged / Wrong / Missing Item  →  Order Management
 1  Delivery Issue  →  Return / Refund
 1  Delivery Issue  →  Account / Access / Security

ERRORS BY AI PREDICTION
 8  Delivery Issue
 2  Other / Unclear
 1  Damaged / Wrong / Missing Item

ERRORS BY HUMAN LABEL
 3  Order Management
 2  Prime Membership
 2  Payment / Billing
 1  Product / Device Support
 1  Delivery Issue
 1  Return / Refund
 1  Account / Access / Security


In [52]:
# ============================================================
# UPDATE HUMAN LABELS AFTER MANUAL ADJUDICATION
# ============================================================

updates = {
    1085150: "Other / Unclear",
    613997: "Other / Unclear",
    1004594: "Delivery Issue",
    1097715: "Other / Unclear",
    132376: "Other / Unclear",
    2500688: "Payment / Billing",
    1376064: "Other / Unclear",
    1540467: "Delivery Issue",
    1135095: "Delivery Issue",
}

# Make sure IDs are comparable
audit["root_tweet_id"] = audit["root_tweet_id"].astype(int)

# Update ONLY the specified human labels
for tweet_id, new_label in updates.items():
    mask = audit["root_tweet_id"] == tweet_id
    if mask.any():
        audit.loc[mask, "human_label"] = new_label
    else:
        print(f"WARNING: ID {tweet_id} not found")

# Save the updated audit
AUDIT_PATH = "../data/golden/pseudo_label_audit_100.csv"
audit.to_csv(AUDIT_PATH, index=False)

print("Updated labels:", len(updates))
print("Saved to:", AUDIT_PATH)
print("\nUpdated rows:")
print(
    audit[audit["root_tweet_id"].isin(updates.keys())]
    [["root_tweet_id", "pseudo_intent", "human_label"]]
    .sort_values("root_tweet_id")
    .to_string(index=False)
)

Updated labels: 9
Saved to: ../data/golden/pseudo_label_audit_100.csv

Updated rows:
 root_tweet_id                  pseudo_intent       human_label
        132376                 Delivery Issue   Other / Unclear
        613997                 Delivery Issue   Other / Unclear
       1004594                 Delivery Issue    Delivery Issue
       1085150                Other / Unclear   Other / Unclear
       1097715 Damaged / Wrong / Missing Item   Other / Unclear
       1135095                 Delivery Issue    Delivery Issue
       1376064                 Delivery Issue   Other / Unclear
       1540467                 Delivery Issue    Delivery Issue
       2500688                 Delivery Issue Payment / Billing


In [53]:
# ============================================================
# FINAL PSEUDO-LABEL AUDIT
# ============================================================

import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

AUDIT_PATH = "../data/golden/pseudo_label_audit_100.csv"

audit_final = pd.read_csv(AUDIT_PATH)

# Clean label strings
audit_final["pseudo_intent"] = audit_final["pseudo_intent"].str.strip()
audit_final["human_label"] = audit_final["human_label"].str.strip()

# Basic validation
assert len(audit_final) == 100, f"Expected 100 rows, found {len(audit_final)}"
assert audit_final["human_label"].notna().all(), "Missing human labels found"

# Compare AI labels against human labels
y_true = audit_final["human_label"]
y_pred = audit_final["pseudo_intent"]

accuracy = accuracy_score(y_true, y_pred)
correct = (y_true == y_pred).sum()
incorrect = (y_true != y_pred).sum()

print("=" * 70)
print("FINAL PSEUDO-LABEL AUDIT")
print("=" * 70)
print(f"Audit examples       : {len(audit_final)}")
print(f"Correct predictions  : {correct}")
print(f"Incorrect predictions: {incorrect}")
print(f"Audit agreement      : {accuracy:.2%}")
print("=" * 70)

# Show remaining disagreements
remaining = audit_final[y_true != y_pred].copy()

print("\nREMAINING DISAGREEMENTS")
print("=" * 70)

for _, row in remaining.iterrows():
    print(f"\nID     : {row['root_tweet_id']}")
    print(f"AI     : {row['pseudo_intent']}")
    print(f"Human  : {row['human_label']}")

print("\nTotal remaining disagreements:", len(remaining))

FINAL PSEUDO-LABEL AUDIT
Audit examples       : 100
Correct predictions  : 93
Incorrect predictions: 7
Audit agreement      : 93.00%

REMAINING DISAGREEMENTS

ID     : 613997
AI     : Delivery Issue
Human  : Other / Unclear

ID     : 2317390
AI     : Other / Unclear
Human  : Delivery Issue

ID     : 1097715
AI     : Damaged / Wrong / Missing Item
Human  : Other / Unclear

ID     : 132376
AI     : Delivery Issue
Human  : Other / Unclear

ID     : 2500688
AI     : Delivery Issue
Human  : Payment / Billing

ID     : 676933
AI     : Delivery Issue
Human  : Prime Membership

ID     : 1376064
AI     : Delivery Issue
Human  : Other / Unclear

Total remaining disagreements: 7


In [54]:
# ============================================================
# FINAL 7 DISAGREEMENTS — COMPACT VIEW
# ============================================================

remaining = audit_final[
    audit_final["pseudo_intent"] != audit_final["human_label"]
].copy()

print("Remaining disagreements:", len(remaining))
print("=" * 100)

print(
    remaining[
        ["root_tweet_id", "pseudo_intent", "human_label"]
    ].to_string(index=False)
)

print("\n" + "=" * 100)
print("ERROR PATTERNS")
print("=" * 100)

print(
    remaining
    .groupby(["pseudo_intent", "human_label"])
    .size()
    .sort_values(ascending=False)
    .to_string()
)

Remaining disagreements: 7
 root_tweet_id                  pseudo_intent       human_label
        613997                 Delivery Issue   Other / Unclear
       2317390                Other / Unclear    Delivery Issue
       1097715 Damaged / Wrong / Missing Item   Other / Unclear
        132376                 Delivery Issue   Other / Unclear
       2500688                 Delivery Issue Payment / Billing
        676933                 Delivery Issue  Prime Membership
       1376064                 Delivery Issue   Other / Unclear

ERROR PATTERNS
pseudo_intent                   human_label      
Delivery Issue                  Other / Unclear      3
Damaged / Wrong / Missing Item  Other / Unclear      1
Delivery Issue                  Payment / Billing    1
                                Prime Membership     1
Other / Unclear                 Delivery Issue       1


In [55]:
# ============================================================
# FINAL 7 DISAGREEMENTS — ADJUDICATION TABLE
# ============================================================

remaining = audit_final[
    audit_final["pseudo_intent"] != audit_final["human_label"]
].copy()

for _, row in remaining.iterrows():
    print("\n" + "=" * 100)
    print(f"ROOT TWEET ID : {row['root_tweet_id']}")
    print("-" * 100)
    print("CONVERSATION:")
    print(row["conversation"])
    print("-" * 100)
    print(f"ORIGINAL HUMAN LABEL : {row['human_label']}")
    print(f"AI LABEL             : {row['pseudo_intent']}")
    print("=" * 100)


ROOT TWEET ID : 613997
----------------------------------------------------------------------------------------------------
CONVERSATION:
CUSTOMER: Amazonの prime now届いたー！
多12時58分に頼んで13時31分出荷、15時13分に受け取りました。ありがたい！わたしは市内住みでは無いのでセンターから10㌔以上離れていますがこの迅速さ。配達の方も礼儀正しくて好印象でした。また急ぎたい時におねがいします。 https://t.co/HBs0H21xwU

AMAZON: @203475 Prime Nowをご利用いただき、ありがとうございました！またお役に立てれば嬉しいです( *´艸｀) EK
----------------------------------------------------------------------------------------------------
ORIGINAL HUMAN LABEL : Other / Unclear
AI LABEL             : Delivery Issue

ROOT TWEET ID : 2317390
----------------------------------------------------------------------------------------------------
CONVERSATION:
CUSTOMER: .@115821 @115850 request you to not use @120169 couriers for deliveries - it's been five days since dox submitted, but shipment not cleared from customs as yet. #Aramex is the most inefficient courier company.

AMAZON: @671878 Apologies for the trouble. I've passed on your feedback to the 

| ID | Conversation | Human Label | AI Label | Comparison | Final Human Label | Rationale |
|---|---|---|---|---|---|---|
| **613997** | Customer says Prime Now arrived quickly and praises the delivery/service. | Other / Unclear | Delivery Issue | AI interpreted delivery-related words as an issue, but the customer reports **no problem**. | **Other / Unclear** | No complaint, delay, or failure is present. Delivery is mentioned positively. |
| **2317390** | Customer says shipment has been stuck in customs for five days and asks Amazon not to use that courier. | Delivery Issue | Other / Unclear | AI missed the explicit shipment/customs delay. | **Delivery Issue** | The primary problem is a shipment not clearing customs and delayed delivery. |
| **1097715** | Customer says "it isn't my order", followed by a cat/joke interaction with Amazon. | Other / Unclear | Damaged / Wrong / Missing Item | AI interpreted "isn't my order" as a wrong-item/order problem, but the thread provides no actionable support context. | **Other / Unclear** | Insufficient evidence to determine a specific support intent; subsequent conversation is unrelated/casual. |
| **132376** | Customer says "Amazon prime will be the death of my bank account." | Other / Unclear | Delivery Issue | AI assigned Delivery Issue without an explicit delivery complaint. | **Other / Unclear** | No delivery problem or specific billing transaction is stated. |
| **2500688** | Customer says they were charged for Prime membership without consent and questions the charge. | Payment / Billing | Delivery Issue | AI missed the explicit unauthorized-charge complaint. | **Payment / Billing** | The primary issue is an unauthorized membership charge. |
| **676933** | Customer says they accidentally became a paid Prime Student member. | Prime Membership | Delivery Issue | AI focused incorrectly on delivery-related patterns rather than membership status. | **Prime Membership** | The explicit issue concerns becoming a paid Prime member. |
| **1376064** | Customer says they cancelled Prime because of Amazon logistics/support but does not describe a specific account problem. | Other / Unclear | Delivery Issue | AI inferred Delivery Issue from "logistics"; human label avoids assuming an unstated specific problem. | **Other / Unclear** | The customer gives a reason for cancellation but does not provide enough detail to assign a specific support intent. |